# 00b — Statistiques descriptives par segment

On reprend la base obtenue à la fin du notebook 00 (périmètre final, dossiers joints unifiés, segment SA / SB) et on décrit les variables séparément dans chaque segment, puisque chaque segment aura son propre modèle.

Pour chaque segment :
1. valeurs manquantes ;
2. variables constantes ou presque ;
3. distributions et valeurs extrêmes ;
4. variables qualitatives ;
5. redondances entre variables ;
6. pouvoir discriminant (IV) ;
7. synthèse : statut provisoire de chaque variable avant la discrétisation, et export Excel.

In [ ]:
import polars as pl
from utils.stats_desc import generer_stats_descriptives, tracer_distributions_numeriques, tracer_taux_defaut_par_variable, tracer_analyse_discrete_avec_defauts

BASE = "data/processed/base_trois_segmentation.parquet"
TARGET, SEGMENT = "ddefaut_ndb", "segment"

base = pl.read_parquet(BASE)
# Filtrage sur le premier segment
seg_nom = sorted(base[SEGMENT].unique())[0]
seg_nom = "SM_multi"
df_seg = base.filter(pl.col(SEGMENT) == seg_nom)

EXCLURE = {"gdt", TARGET, "datdelhis", SEGMENT, "has_engagement", "SelectionProb", "SamplingWeight", "type_PM_PP", "code_marche_v2"}
VARS_NUM = [c for c, t in df_seg.schema.items() if c not in EXCLURE and t.is_numeric()]
VARS_QUALI = [c for c, t in df_seg.schema.items() if c not in EXCLURE and not t.is_numeric() and not t.is_temporal()]

print(f"Segment : {seg_nom} | Lignes : {df_seg.height} | Cible moyenne : {df_seg[TARGET].mean():.2%}")

# Analyse des valeurs manquantes

In [ ]:
stats_df = generer_stats_descriptives(df_seg, VARS_NUM, VARS_QUALI)
stats_df

In [ ]:
# 1. Histogrammes des variables quantitatives
tracer_distributions_numeriques(df_seg, VARS_NUM[1:10])  # 12 premières pour illustration

# 2. Taux de défaut et effectifs par modalité
tracer_taux_defaut_par_variable(df_seg, VARS_NUM[:6] + VARS_QUALI[:2], target=TARGET)

In [ ]:
# Sélection automatique ou manuelle des variables discrètes (ex: moins de 15 modalités uniques)
VARS_DISCRETES = [c for c in VARS_NUM if df_seg[c].n_unique() <= 15] + VARS_QUALI

# Affichage des graphiques avec effectif, nombre de défauts et taux de défaut
tracer_analyse_discrete_avec_defauts(
    df_seg, 
    vars_discretes=VARS_DISCRETES, 
    target=TARGET,
    vars_exclues=["AGE_PM_EN_MOIS","MNT_HA102", "TOTAL_LITIGES_PRO_IND_0209", "TOTAL_TITRES_PRO_IND_0205", "CRTOP_AG_IND_0038","nb_credit_perso", "nb_credit_flex", "nb_credit_LCP", "nb_credit_revolving", "nb_credit_tresorerie", "Dfo_max_Mm1_Mm3","LISTE_CODNAF2", "LISTE_CODETAJUR", "LISTE_CODACVPRO_modif", "RATIO_LITIGES_vs_CA", "TOTAL_EPS_PRO_IND_0208","NBJ_ARR_SS", "ARR_GDT", "nb_PGE", "nb_Neiertz", "nb_credit_garantie_emise", "nb_credit_autre", "nb_credit_bail","engagement", "nb_tier", "bilan", "hors_bilan", "nb_credit_immo","nb_credit_immo__sans_garantie", "nb_credit_immo__crelog","nb_credit_immo__hypotheque", "nb_credit_MLT", "nb_credit_MLT__sans_garantie"]
)